In [1]:
import os

repo_url = "https://github.com/MrX9955/Kidney-Disease-MLOps.git"
repo_path = "/content/Kidney-Disease-MLOps"

if not os.path.exists(repo_path):
    !git clone {repo_url} {repo_path}
else:
    print("Repository already exists!")

os.chdir(repo_path)

print("Repository connected successfully!")
print("Current directory:", os.getcwd())

Cloning into '/content/Kidney-Disease-MLOps'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 75 (delta 17), reused 47 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (75/75), 8.64 MiB | 11.24 MiB/s, done.
Resolving deltas: 100% (17/17), done.
Repository connected successfully!
Current directory: /content/Kidney-Disease-MLOps


In [2]:
import pandas as pd
import os

dataset_path = "datasets/v2/kidney_disease_v2.csv"

df = pd.read_csv(dataset_path)

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

display(df.head())

print("\nMissing Values:")
print(df.isnull().sum())

Dataset loaded successfully!
Dataset Shape: (1500, 13)


,Age,Blood_Pressure,Blood_Glucose,Creatinine,Hemoglobin,BMI,Urea,Sodium,Gender,Smoking,Diabetes,Hypertension,Disease_Risk
0,63,106.77,107.56,1.40,13.68,25.26,37.87,132.51,Female,Yes,Yes,No,Low_Risk
1,41,132.74,87.31,0.99,14.31,19.67,23.06,135.29,Female,Yes,No,Yes,No_Disease
2,27,109.43,102.27,0.80,15.26,28.83,20.63,137.68,Female,Yes,No,No,No_Disease
3,64,141.70,187.58,3.39,7.85,31.87,59.09,139.46,Female,Yes,Yes,Yes,High_Risk
4,63,89.16,114.84,0.90,16.22,22.55,22.93,141.78,Male,No,No,Yes,No_Disease



Missing Values:
Age                0
Blood_Pressure    45
Blood_Glucose     45
Creatinine         0
Hemoglobin         0
BMI               45
Urea               0
Sodium             0
Gender             0
Smoking           45
Diabetes           0
Hypertension       0
Disease_Risk       0
dtype: int64


In [3]:
folders = [
    "experiment_2",
    "experiment_2/pipelines",
    "experiment_2/features",
    "experiment_2/feature_store",
    "experiment_2/outputs",
    "experiment_2/screenshots"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Experiment 2 folders created successfully!")

Experiment 2 folders created successfully!


In [4]:
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

print("Preprocessing libraries imported successfully!")

Preprocessing libraries imported successfully!


In [5]:
# Separate input features and target
X = df.drop("Disease_Risk", axis=1)
y = df["Disease_Risk"]

# Identify numerical and categorical columns
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical Features:", numeric_features)
print("Categorical Features:", categorical_features)

print("Target Classes:", y.nunique())

Numerical Features: ['Age', 'Blood_Pressure', 'Blood_Glucose', 'Creatinine', 'Hemoglobin', 'BMI', 'Urea', 'Sodium']
Categorical Features: ['Gender', 'Smoking', 'Diabetes', 'Hypertension']
Target Classes: 5


In [6]:
# Numerical preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

# Combine both pipelines
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Automated preprocessing pipeline created!")

Automated preprocessing pipeline created!


In [7]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Fit only on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform testing data
X_test_processed = preprocessor.transform(X_test)

print("Data preprocessing completed!")
print("Training Shape:", X_train_processed.shape)
print("Testing Shape:", X_test_processed.shape)

Data preprocessing completed!
Training Shape: (1200, 16)
Testing Shape: (300, 16)


In [8]:
os.makedirs("experiment_2/pipelines", exist_ok=True)

joblib.dump(
    preprocessor,
    "experiment_2/pipelines/preprocessor.pkl"
)

print("Preprocessing pipeline saved successfully!")

Preprocessing pipeline saved successfully!


In [9]:
print("Training Shape:", X_train_processed.shape)
print("Testing Shape:", X_test_processed.shape)
print("Missing values after preprocessing:",
      np.isnan(X_train_processed).sum())

Training Shape: (1200, 16)
Testing Shape: (300, 16)
Missing values after preprocessing: 0


In [12]:
# Create a copy of the original dataset
feature_df = df.copy()

# Feature 1: Kidney function indicator
feature_df["Kidney_Function_Index"] = (
    feature_df["Creatinine"] + feature_df["Urea"] / 100
)

# Feature 2: Blood health indicator
feature_df["Blood_Health_Index"] = (
    feature_df["Hemoglobin"] / feature_df["Blood_Glucose"]
)

# Feature 3: Lifestyle risk indicator
feature_df["Lifestyle_Risk"] = (
    (feature_df["Smoking"] == "Yes").astype(int) +
    (feature_df["Diabetes"] == "Yes").astype(int) +
    (feature_df["Hypertension"] == "Yes").astype(int)
)

print("Feature engineering completed!")
display(feature_df.head())

print("\nNew Features:")
print([
    "Kidney_Function_Index",
    "Blood_Health_Index",
    "Lifestyle_Risk"
])

Feature engineering completed!


,Age,Blood_Pressure,Blood_Glucose,Creatinine,Hemoglobin,BMI,Urea,Sodium,Gender,Smoking,Diabetes,Hypertension,Disease_Risk,Kidney_Function_Index,Blood_Health_Index,Lifestyle_Risk
0,63,106.77,107.56,1.40,13.68,25.26,37.87,132.51,Female,Yes,Yes,No,Low_Risk,1.7787,0.127185,2
1,41,132.74,87.31,0.99,14.31,19.67,23.06,135.29,Female,Yes,No,Yes,No_Disease,1.2206,0.163899,2
2,27,109.43,102.27,0.80,15.26,28.83,20.63,137.68,Female,Yes,No,No,No_Disease,1.0063,0.149213,1
3,64,141.70,187.58,3.39,7.85,31.87,59.09,139.46,Female,Yes,Yes,Yes,High_Risk,3.9809,0.041849,3
4,63,89.16,114.84,0.90,16.22,22.55,22.93,141.78,Male,No,No,Yes,No_Disease,1.1293,0.141240,1



New Features:
['Kidney_Function_Index', 'Blood_Health_Index', 'Lifestyle_Risk']


In [13]:
os.makedirs("experiment_2/features", exist_ok=True)

feature_df.to_csv(
    "experiment_2/features/engineered_features.csv",
    index=False
)

print("Engineered features saved successfully!")
print("Feature dataset shape:", feature_df.shape)

Engineered features saved successfully!
Feature dataset shape: (1500, 16)


In [14]:
import json
import os

# Create feature store directory
store_path = "experiment_2/feature_store"
os.makedirs(store_path, exist_ok=True)

# Create unique record IDs
feature_store_df = feature_df.drop(
    columns=["Disease_Risk"]
).copy()

feature_store_df.insert(
    0, "record_id",
    range(1, len(feature_store_df) + 1)
)

# Save features
feature_store_df.to_csv(
    f"{store_path}/features.csv",
    index=False
)

# Save target labels separately
labels_df = pd.DataFrame({
    "record_id": range(1, len(df) + 1),
    "Disease_Risk": df["Disease_Risk"]
})

labels_df.to_csv(
    f"{store_path}/labels.csv",
    index=False
)

# Create feature metadata
metadata = {
    "feature_store": "Kidney Disease Feature Store",
    "total_records": len(feature_store_df),
    "total_features": len(feature_store_df.columns) - 1,
    "feature_names": feature_store_df.columns[1:].tolist(),
    "storage_format": "CSV"
}

with open(f"{store_path}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Feature Store created successfully!")
print("Total Records:", len(feature_store_df))
print("Total Features:", len(feature_store_df.columns) - 1)

Feature Store created successfully!
Total Records: 1500
Total Features: 15


In [15]:
# Load features from Feature Store
stored_features = pd.read_csv(
    "experiment_2/feature_store/features.csv"
)

# Retrieve first five records
retrieved_data = stored_features.head(5)

print("Features retrieved successfully!")
display(retrieved_data)

Features retrieved successfully!


,record_id,Age,Blood_Pressure,Blood_Glucose,Creatinine,Hemoglobin,BMI,Urea,Sodium,Gender,Smoking,Diabetes,Hypertension,Kidney_Function_Index,Blood_Health_Index,Lifestyle_Risk
0,1,63,106.77,107.56,1.40,13.68,25.26,37.87,132.51,Female,Yes,Yes,No,1.7787,0.127185,2
1,2,41,132.74,87.31,0.99,14.31,19.67,23.06,135.29,Female,Yes,No,Yes,1.2206,0.163899,2
2,3,27,109.43,102.27,0.80,15.26,28.83,20.63,137.68,Female,Yes,No,No,1.0063,0.149213,1
3,4,64,141.70,187.58,3.39,7.85,31.87,59.09,139.46,Female,Yes,Yes,Yes,3.9809,0.041849,3
4,5,63,89.16,114.84,0.90,16.22,22.55,22.93,141.78,Male,No,No,Yes,1.1293,0.141240,1


In [16]:
print("Feature Store Verification")

print("Total Records:", stored_features.shape[0])
print("Total Columns:", stored_features.shape[1])

print("Missing Values:")
print(stored_features.isnull().sum())

print("\nFeature Store Status: Verified")

Feature Store Verification
Total Records: 1500
Total Columns: 16
Missing Values:
record_id                 0
Age                       0
Blood_Pressure           45
Blood_Glucose            45
Creatinine                0
Hemoglobin                0
BMI                      45
Urea                      0
Sodium                    0
Gender                    0
Smoking                  45
Diabetes                  0
Hypertension              0
Kidney_Function_Index     0
Blood_Health_Index       45
Lifestyle_Risk            0
dtype: int64

Feature Store Status: Verified


In [17]:
import os

for root, dirs, files in os.walk("experiment_2"):
    for file in files:
        print(os.path.join(root, file))

experiment_2/feature_store/metadata.json
experiment_2/feature_store/labels.csv
experiment_2/feature_store/features.csv
experiment_2/features/engineered_features.csv
experiment_2/pipelines/preprocessor.pkl


In [18]:
import os

os.chdir("/content/Kidney-Disease-MLOps")

!git status
!git branch --show-current
!git remote -v

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	experiment_2/

nothing added to commit but untracked files present (use "git add" to track)
main
origin	https://github.com/MrX9955/Kidney-Disease-MLOps.git (fetch)
origin	https://github.com/MrX9955/Kidney-Disease-MLOps.git (push)


In [19]:
%cd /content/Kidney-Disease-MLOps

!git add experiment_2/

!git status

/content
On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   experiment_2/feature_store/features.csv
	new file:   experiment_2/feature_store/labels.csv
	new file:   experiment_2/feature_store/metadata.json
	new file:   experiment_2/features/engineered_features.csv
	new file:   experiment_2/pipelines/preprocessor.pkl



In [20]:
!git commit -m "Complete Experiment 2 preprocessing and feature store"

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@cbc92c05206e.(none)')


In [21]:
!git config --global user.name "Vikash Kumar Sah"
!git config --global user.email "231fa04f67@gmail.com"

In [22]:
!git commit -m "Complete Experiment 2 preprocessing and feature store"

[main d2721ec] Complete Experiment 2 preprocessing and feature store
 5 files changed, 4526 insertions(+)
 create mode 100644 experiment_2/feature_store/features.csv
 create mode 100644 experiment_2/feature_store/labels.csv
 create mode 100644 experiment_2/feature_store/metadata.json
 create mode 100644 experiment_2/features/engineered_features.csv
 create mode 100644 experiment_2/pipelines/preprocessor.pkl


In [23]:
!git push origin main

fatal: could not read Username for 'https://github.com': No such device or address


In [24]:
from getpass import getpass
import os

token = getpass("Enter your GitHub PAT: ")

os.environ["GH_TOKEN"] = token

print("Token received securely!")

Enter your GitHub PAT: ··········
Token received securely!


In [25]:
import subprocess

repo_url = "https://github.com/MrX9955/Kidney-Disease-MLOps.git"

authenticated_url = repo_url.replace(
    "https://",
    f"https://{os.environ['GH_TOKEN']}@"
)

subprocess.run(
    ["git", "push", authenticated_url, "main"],
    check=True
)

print("Experiment 2 uploaded successfully!")

Experiment 2 uploaded successfully!
